# 08 — Comparaison des modeles & analyse du sur-apprentissage

**Objectif (exigence mentor)** : comparer tous les modeles via le **F1-score**, tracer les **courbes d'apprentissage** (diagnostic du sur-apprentissage) et les matrices de confusion.

**Correctif methodologique critique** : on re-score l'hybride (CNN+GB) sur le **meme test set fige** que les modeles TL avant comparaison.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# --- Portabilite Colab / local ---
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Montez votre Drive et placez-y le projet, puis ajustez ce chemin.
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/covid_19_radiographie"  # <-- a adapter
    os.environ["COVID_BASE_DIR"] = PROJECT_DIR
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
    # Installe les dependances manquantes sur Colab
    !pip -q install shap
else:
    # En local : racine = parent du dossier notebooks/
    PROJECT_DIR = os.path.abspath("..")
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)

from src.utils.env import setup_environment, ensure_dirs, get_paths
INFO = setup_environment()
PATHS = ensure_dirs()
print("Projet :", PROJECT_DIR)

## 1. Re-scoring de l'hybride (CNN + Gradient Boosting) sur le test set fige

Le modele hybride d'origine a ete evalue sur un split 80/20 different : ses chiffres ne sont pas comparables. On le re-evalue sur `reports/splits/test.csv`.

*(Le baseline RF-HOG n'a pas de modele `.pkl` sauvegarde ; pour l'inclure de maniere comparable il faut le re-entrainer sur le meme split via `src/models/train_model.py`. Ses chiffres d'origine restent indicatifs.)*

In [ ]:
import numpy as np, joblib, cv2
from PIL import Image
from tensorflow.keras.models import Model, load_model
from src.data import tf_pipeline as tp
from src.models import evaluation as ev

HYBRID_PKL = PATHS['models'] / 'hybrid_boosting.pkl'
CNN_PATH   = PATHS['reports'] / 'best_model.keras'

def rescore_hybrid():
    artifact = joblib.load(HYBRID_PKL)
    gb = artifact['gb_model']; classes = artifact['classes']
    cnn = load_model(str(CNN_PATH), compile=False)
    feat = Model(cnn.input, cnn.get_layer(artifact['feature_layer']).output)
    # Meme preprocessing que build_hybrid_model (256->/255->128->/255, grayscale)
    def prep(fp):
        im = Image.open(fp).convert('L').resize((256, 256))
        a = np.array(im, dtype=np.float32) / 255.0
        a = cv2.resize(a, (128, 128)).astype(np.float32) / 255.0
        return a[..., np.newaxis]
    test_df = tp.load_split_df('test')
    # alignement de l'ordre des classes hybride -> indices canoniques
    idx_map = {i: tp.CLASS_TO_IDX[c] for i, c in enumerate(classes)}
    X = np.stack([prep(fp) for fp in test_df['filepath']])
    F = feat.predict(X, batch_size=128, verbose=0)
    proba_raw = gb.predict_proba(F)
    # reordonne les colonnes vers l'ordre canonique CLASS_NAMES
    proba = np.zeros_like(proba_raw)
    for i in range(proba_raw.shape[1]):
        proba[:, idx_map[i]] = proba_raw[:, i]
    y_true = test_df['label'].to_numpy()
    y_pred = proba.argmax(axis=1)
    m = ev.evaluate_model('hybride_cnn_gb', y_true, y_pred, proba, save_dir=PATHS['metrics'])
    ev.print_metrics_summary(m); return m

if HYBRID_PKL.exists() and CNN_PATH.exists():
    hybrid_metrics = rescore_hybrid()
else:
    print('Artefacts hybride absents — re-scoring saute.')

## 2. Table maitresse de comparaison

Assemble tous les `*_test_metrics.json` (TL + hybride re-score) en une table triee par F1-macro.

In [ ]:
from src.models import comparison as cmp
metrics_list = cmp.load_all_metrics()
table = cmp.build_comparison_table(metrics_list)
cmp.save_comparison_table(table)
display(table)

## 3. Courbes d'apprentissage (sur-apprentissage)

Train vs validation (loss + accuracy), phases 1 et 2. L'**ecart train-val** diagnostique le sur-apprentissage ; la ligne pointillee marque le debut du fine-tuning.

In [ ]:
for b in ['efficientnetb0', 'resnet50', 'inceptionv3']:
    try:
        cmp.plot_learning_curves(b, save_path=PATHS['figures'] / f'learning_curves_{b}.png')
    except FileNotFoundError as e:
        print(e)

## 4. Matrices de confusion cote a cote

In [ ]:
cmp.plot_confusion_grid(metrics_list, save_path=PATHS['figures'] / 'comparison_confusions.png')

## 5. Discussion (a reporter dans le rapport de synthese)

- **Meilleur modele** selon F1-macro et selon le rappel COVID (garde-fou clinique).
- **Sur-apprentissage** : ampleur de l'ecart train-val, epoque de declenchement de l'EarlyStopping, effet du fine-tuning (phase 2).
- **Le Transfer Learning bat-il l'hybride (F1-macro 0.84) ?** Compromis cout/performance (EfficientNetB0 vs InceptionV3).
- **Confusions residuelles** : classes les plus confondues et interpretation clinique.